# Energy Advisor Validation Workbook

This notebook is a validation and exploration companion for the Home Energy System project.

It covers:
- dummy dynamic electricity price generation (24h, 7d, 8760)
- hourly consumption profile construction
- simulation sanity checks
- basic optimization and trade-off visualization

In [1]:
import os
import sys
from dataclasses import dataclass

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Make project modules importable when notebook runs from notebooks/
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from models import (
    SystemConfig, SolarConfig, BatteryConfig, EVConfig,
    HeatPumpConfig, AircoConfig, HouseholdConfig, EconomicsConfig,
)
from simulation import simulate, TOU_PRICES
from optimizer import optimise

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 140)
print('Imports ready')

Imports ready


## 1) Dummy Dynamic Electricity Prices

A practical default for dynamic pricing is a synthetic hourly series for a full year (8760 points).

The pattern below combines:
- daily shape (morning/evening peaks)
- seasonal shape (winter higher, summer lower)
- weekend discount
- random but bounded volatility

In [2]:
def generate_dummy_dynamic_prices(year: int = 2025, seed: int = 42, base_eur_per_kwh: float = 0.28) -> pd.Series:
    """Create a synthetic but realistic hourly import price series for one year."""
    idx = pd.date_range(f'{year}-01-01', f'{year+1}-01-01', freq='h', inclusive='left')
    rng = np.random.default_rng(seed)

    hour = idx.hour.to_numpy()
    doy = idx.dayofyear.to_numpy()
    dow = idx.dayofweek.to_numpy()

    morning_peak = np.exp(-0.5 * ((hour - 8) / 2.0) ** 2)
    evening_peak = np.exp(-0.5 * ((hour - 19) / 2.5) ** 2)
    daily_shape = 0.05 * morning_peak + 0.10 * evening_peak

    season_shape = 0.06 * np.cos(2 * np.pi * (doy - 15) / 365.0)

    weekend_discount = np.where(dow >= 5, -0.015, 0.0)

    noise = rng.normal(0.0, 0.012, size=len(idx))

    prices = base_eur_per_kwh + daily_shape + season_shape + weekend_discount + noise
    prices = np.clip(prices, 0.05, 0.70)

    return pd.Series(prices, index=idx, name='import_price_eur_per_kwh')

dynamic_8760 = generate_dummy_dynamic_prices()
dynamic_8760.describe()

count    8760.000000
mean        0.311293
std         0.053925
min         0.174152
25%         0.268636
50%         0.312676
75%         0.349788
max         0.474248
Name: import_price_eur_per_kwh, dtype: float64

In [3]:
sample_week = dynamic_8760.loc['2025-01-06':'2025-01-12 23:00']
fig = px.line(sample_week, title='Dummy Dynamic Prices - Sample Week', labels={'value': 'EUR/kWh', 'index': 'Time'})
fig.update_layout(height=320)
fig.show()

monthly = dynamic_8760.resample('ME').mean()
fig_m = px.bar(monthly, title='Average Monthly Import Price (Dummy Dynamic)', labels={'value': 'EUR/kWh', 'index': 'Month'})
fig_m.update_layout(height=320)
fig_m.show()

### Price Input Fallback Rule (24h / 7d / 8760)

If user data is shorter than a full year, upscale it:
- 24h: repeat daily for full horizon
- 7d (168h): repeat weekly
- 8760h: use directly
- missing: fallback to fixed flat price or default synthetic dynamic series

In [4]:
def expand_price_series(raw: np.ndarray, target_len: int) -> np.ndarray:
    raw = np.asarray(raw, dtype=float)
    if raw.ndim != 1 or len(raw) == 0:
        raise ValueError('Price input must be a non-empty 1D array')
    reps = int(np.ceil(target_len / len(raw)))
    return np.tile(raw, reps)[:target_len]

def resolve_import_prices(
    contract_type: str,
    horizon_hours: int,
    flat_price: float = 0.28,
    tou_24h: np.ndarray | None = None,
    dynamic_hourly: np.ndarray | None = None,
) -> np.ndarray:
    contract = contract_type.lower()
    if contract == 'fixed':
        return np.full(horizon_hours, flat_price, dtype=float)

    if contract == 'tou':
        source = TOU_PRICES if tou_24h is None else np.asarray(tou_24h, dtype=float)
        return expand_price_series(source, horizon_hours)

    if contract == 'dynamic':
        if dynamic_hourly is None:
            dyn = generate_dummy_dynamic_prices().to_numpy()
            return expand_price_series(dyn, horizon_hours)
        return expand_price_series(dynamic_hourly, horizon_hours)

    raise ValueError(f'Unsupported contract_type: {contract_type}')

print('24h from TOU:', resolve_import_prices('tou', 24)[:5])
print('168h from TOU:', resolve_import_prices('tou', 168)[:5])
print('8760h fixed mean:', resolve_import_prices('fixed', 8760).mean())
print('8760h dynamic mean:', resolve_import_prices('dynamic', 8760).mean())

24h from TOU: [0.18 0.18 0.18 0.18 0.18]
168h from TOU: [0.18 0.18 0.18 0.18 0.18]
8760h fixed mean: 0.28
8760h dynamic mean: 0.31129319588797705


## 2) Hourly Consumption Profiles

The app currently uses deterministic profile functions. For advisor mode, this notebook exposes behavior-driven profile generation so assumptions are inspectable.

In [5]:
@dataclass
class BehaviorInputs:
    household_kwh_day: float = 15.0
    heat_pump_kwh_day: float = 8.0
    ev_kwh_day: float = 8.5
    dhw_kwh_day: float = 2.0
    wfh_days_per_week: int = 2
    ev_arrival_hour: int = 18
    ev_departure_hour: int = 7
    ev_target_kwh_at_departure: float = 8.5

def normalize_shape(shape: np.ndarray) -> np.ndarray:
    shape = np.asarray(shape, dtype=float)
    total = shape.sum()
    return shape / total if total > 0 else np.full_like(shape, 1 / len(shape))

def base_household_shape() -> np.ndarray:
    return normalize_shape(np.array([
        0.6, 0.5, 0.45, 0.45, 0.5, 0.7,
        1.0, 1.1, 0.95, 0.8, 0.75, 0.75,
        0.8, 0.85, 0.8, 0.85, 1.0, 1.3,
        1.6, 1.8, 1.7, 1.4, 1.1, 0.8
    ]))

def build_weekly_profiles(behavior: BehaviorInputs, temp_c: float = 12.0, sun_factor: float = 1.0) -> pd.DataFrame:
    idx = pd.date_range('2025-01-06', periods=24*7, freq='h')
    hour = idx.hour
    dow = idx.dayofweek

    # Household base + simple WFH uplift on selected weekdays
    hh_day = behavior.household_kwh_day
    hh_shape = base_household_shape()
    household = np.zeros(len(idx), dtype=float)
    wfh_weekdays = set(range(0, min(behavior.wfh_days_per_week, 5)))
    for d in range(7):
        mask = dow == d
        uplift = 1.10 if d in wfh_weekdays else 1.0
        weekend = 1.08 if d >= 5 else 1.0
        household[mask] = hh_day * uplift * weekend * hh_shape

    # Heat pump: more demand when colder
    hp_temp_mult = float(np.clip(1.0 + (18 - temp_c) * 0.03, 0.5, 2.2))
    hp_daily = behavior.heat_pump_kwh_day * hp_temp_mult
    hp_shape = np.zeros(24)
    hp_shape[6:9] = 1.2
    hp_shape[9:17] = 0.4
    hp_shape[17:21] = 1.5
    hp_shape[21:24] = 0.2
    hp_shape = normalize_shape(hp_shape)
    heat_pump = np.tile(hp_daily * hp_shape, 7)

    # DHW: morning and evening draws
    dhw_shape = np.zeros(24)
    dhw_shape[6:9] = 1.4
    dhw_shape[18:22] = 1.6
    dhw_shape = normalize_shape(dhw_shape)
    dhw = np.tile(behavior.dhw_kwh_day * dhw_shape, 7)

    # EV: only charge when car is at home
    ev = np.zeros(len(idx), dtype=float)
    for i, ts in enumerate(idx):
        h = ts.hour
        at_home = (h >= behavior.ev_arrival_hour) or (h < behavior.ev_departure_hour)
        if at_home:
            ev[i] = 1.0
    # Convert availability mask to kWh profile
    ev_shape_day = normalize_shape(ev[:24])
    ev_daily = behavior.ev_kwh_day
    ev = np.tile(ev_daily * ev_shape_day, 7)

    # Solar: daytime bell shape scaled by sun factor
    solar_shape = np.array([
        0.00,0.00,0.00,0.00,0.00,0.02,
        0.06,0.12,0.20,0.28,0.35,0.38,
        0.38,0.35,0.28,0.20,0.12,0.06,
        0.02,0.00,0.00,0.00,0.00,0.00
    ])
    solar_shape = normalize_shape(solar_shape)
    solar_kwh_day = 20.0 * float(np.clip(sun_factor, 0.2, 2.2))
    solar = np.tile(solar_kwh_day * solar_shape, 7)

    df = pd.DataFrame({
        'household': household,
        'heat_pump': heat_pump,
        'dhw': dhw,
        'ev': ev,
        'solar': solar,
    }, index=idx)
    df['total_demand'] = df['household'] + df['heat_pump'] + df['dhw'] + df['ev']
    return df

behavior = BehaviorInputs()
profiles_week = build_weekly_profiles(behavior, temp_c=5, sun_factor=0.8)
profiles_week.head()

,household,heat_pump,dhw,ev,solar,total_demand
2025-01-06 00:00:00,0.439024,0.0,0.0,0.653846,0.0,1.092871
2025-01-06 01:00:00,0.365854,0.0,0.0,0.653846,0.0,1.019700
2025-01-06 02:00:00,0.329268,0.0,0.0,0.653846,0.0,0.983114
2025-01-06 03:00:00,0.329268,0.0,0.0,0.653846,0.0,0.983114
2025-01-06 04:00:00,0.365854,0.0,0.0,0.653846,0.0,1.019700


In [6]:
view = profiles_week[['household', 'heat_pump', 'dhw', 'ev', 'solar']].copy()
fig = go.Figure()
for col in ['household', 'heat_pump', 'dhw', 'ev']:
    fig.add_trace(go.Scatter(x=view.index, y=view[col], name=col, stackgroup='demand'))
fig.add_trace(go.Scatter(x=view.index, y=view['solar'], name='solar', line=dict(width=2)))
fig.update_layout(title='Weekly Hourly Profiles (Demand Components + Solar)', height=380, xaxis_title='Time', yaxis_title='kWh/h')
fig.show()

daily_totals = profiles_week.resample('D').sum()[['total_demand', 'solar']]
daily_totals

,total_demand,solar
2025-01-06,38.12,16.0
2025-01-07,38.12,16.0
2025-01-08,36.62,16.0
2025-01-09,36.62,16.0
2025-01-10,36.62,16.0
2025-01-11,37.82,16.0
2025-01-12,37.82,16.0


## 3) Simulation Calculation Principles

Run the project simulation for a selected day and inspect energy flows, battery behavior, and costs.

In [7]:
cfg = SystemConfig(
    solar=SolarConfig(kwp=8.0, efficiency=0.85),
    battery=BatteryConfig(capacity_kwh=12.0, max_charge_kw=4.0, max_discharge_kw=4.0, initial_soc_pct=20, enabled=True),
    ev=EVConfig(enabled=True, weekly_kwh=60.0, flexible=True),
    heat_pump=HeatPumpConfig(enabled=True, daily_kwh=9.0),
    airco=AircoConfig(enabled=False, daily_kwh=0.0, intensity=1.0),
    household=HouseholdConfig(base_kwh_day=14.0, peak_multiplier=2.0),
    economics=EconomicsConfig(import_price=0.30, feedin_tariff=0.08, time_of_use=True),
)

res_base = simulate(cfg)
res_opt = simulate(optimise(cfg))

summary = pd.DataFrame({
    'base': {
        'daily_cost': res_base['daily_cost_with_system'],
        'daily_savings_vs_baseline': res_base['daily_savings'],
        'self_consumption_pct': res_base['self_consumption_pct'],
        'grid_dependency_pct': res_base['grid_dependency_pct'],
    },
    'optimized': {
        'daily_cost': res_opt['daily_cost_with_system'],
        'daily_savings_vs_baseline': res_opt['daily_savings'],
        'self_consumption_pct': res_opt['self_consumption_pct'],
        'grid_dependency_pct': res_opt['grid_dependency_pct'],
    }
})
summary

,base,optimized
daily_cost,-1.651313,-2.227313
daily_savings_vs_baseline,12.407860,12.983860
self_consumption_pct,59.096079,44.828244
grid_dependency_pct,0.000000,0.000000


In [8]:
hours = np.arange(24)
fig = go.Figure()
fig.add_trace(go.Bar(x=hours, y=res_base['grid_import'], name='grid_import_base'))
fig.add_trace(go.Bar(x=hours, y=res_base['grid_export'], name='grid_export_base'))
fig.add_trace(go.Scatter(x=hours, y=res_base['battery_soc'], name='battery_soc_base', yaxis='y2'))
fig.update_layout(
    title='Simulation Flows for One Day',
    xaxis_title='Hour',
    yaxis_title='kWh/h',
    yaxis2=dict(title='Battery SOC (kWh)', overlaying='y', side='right'),
    barmode='group',
    height=360,
)
fig.show()

## 4) Optimization Sweep and Trade-offs

Small sweep to validate ranking behavior. Expand this grid once runtime is acceptable.

In [9]:
def estimate_capex(solar_kwp: float, battery_kwh: float, p_solar: float = 1250, p_battery: float = 550) -> float:
    return solar_kwp * p_solar + battery_kwh * p_battery

def evaluate_candidates(
    solar_range=range(0, 21, 2),
    battery_range=range(0, 21, 2),
    import_price=0.30,
    feedin_tariff=0.08,
):
    rows = []
    for s in solar_range:
        for b in battery_range:
            cfg = SystemConfig(
                solar=SolarConfig(kwp=float(s), efficiency=0.85),
                battery=BatteryConfig(
                    capacity_kwh=float(b),
                    max_charge_kw=max(0.5, b/3) if b > 0 else 0.0,
                    max_discharge_kw=max(0.5, b/3) if b > 0 else 0.0,
                    initial_soc_pct=20,
                    enabled=b > 0,
                ),
                ev=EVConfig(enabled=True, weekly_kwh=60.0, flexible=True),
                heat_pump=HeatPumpConfig(enabled=True, daily_kwh=9.0),
                airco=AircoConfig(enabled=False, daily_kwh=0.0, intensity=1.0),
                household=HouseholdConfig(base_kwh_day=14.0, peak_multiplier=2.0),
                economics=EconomicsConfig(import_price=import_price, feedin_tariff=feedin_tariff, time_of_use=True),
            )
            out = simulate(cfg)
            annual_savings = out['daily_savings'] * 365
            capex = estimate_capex(float(s), float(b))
            payback = np.inf if annual_savings <= 0 else capex / annual_savings
            roi_score = annual_savings - 0.1 * capex
            rows.append({
                'solar_kwp': float(s),
                'battery_kwh': float(b),
                'annual_savings_eur': float(annual_savings),
                'capex_eur': float(capex),
                'payback_years': float(payback),
                'roi_score': float(roi_score),
            })
    return pd.DataFrame(rows)

cand = evaluate_candidates()
cand.sort_values('annual_savings_eur', ascending=False).head(10)

,solar_kwp,battery_kwh,annual_savings_eur,capex_eur,payback_years,roi_score
115,20.0,10.0,6732.382717,30500.0,4.530343,3682.382717
116,20.0,12.0,6685.662717,31600.0,4.726532,3525.662717
117,20.0,14.0,6638.942717,32700.0,4.925483,3368.942717
114,20.0,8.0,6616.868556,29400.0,4.443189,3676.868556
118,20.0,16.0,6592.222717,33800.0,5.127254,3212.222717
119,20.0,18.0,6545.502717,34900.0,5.331905,3055.502717
120,20.0,20.0,6498.782717,36000.0,5.539499,2898.782717
104,18.0,10.0,6366.614296,28000.0,4.397942,3566.614296
113,20.0,6.0,6364.827031,28300.0,4.446311,3534.827031
105,18.0,12.0,6319.894296,29100.0,4.604507,3409.894296


In [10]:
best_savings = cand.loc[cand['annual_savings_eur'].idxmax()]
best_payback = cand.replace({'payback_years': {np.inf: np.nan}}).loc[cand.replace({'payback_years': {np.inf: np.nan}})['payback_years'].idxmin()]
best_roi = cand.loc[cand['roi_score'].idxmax()]

display(pd.DataFrame([best_savings, best_payback, best_roi], index=['best_annual_savings', 'best_payback', 'best_roi_score']))

fig = px.scatter(
    cand,
    x='capex_eur',
    y='annual_savings_eur',
    color='roi_score',
    hover_data=['solar_kwp', 'battery_kwh', 'payback_years'],
    title='Candidate Trade-off: Capex vs Annual Savings',
)
fig.update_layout(height=360)
fig.show()

,solar_kwp,battery_kwh,annual_savings_eur,capex_eur,payback_years,roi_score
best_annual_savings,20.0,10.0,6732.382717,30500.0,4.530343,3682.382717
best_payback,2.0,0.0,1387.193039,2500.0,1.802201,1137.193039
best_roi_score,20.0,10.0,6732.382717,30500.0,4.530343,3682.382717


## 5) Validation Checklist

Use this checklist before integrating changes into app code:
- price fallback behaves correctly for 24h, 7d, 8760
- profile totals match configured daily energy
- temperature changes move HP demand in expected direction
- optimization objective changes the recommended configuration
- recommendation explanation references simulation evidence